# CLIP-ViT + Transformer — Dynamic Logits + Contrastive Loss (không HalluLoss)

Notebook này nhân bản từ `coco2014-clip-vit-contrastive.ipynb` (xem `research/notebooks/manifest.csv`,
`N009`), giữ nguyên Dynamic Decoder Intercept và Contrastive loss (InfoNCE, xem
`research/contrastive-loss-for-caption-hallucination.md`), nhưng **tắt HalluLoss** theo yêu cầu.

## Cấu hình

```text
USE_HALL_LOSS = False              # tắt hoàn toàn - hall_loss KHÔNG được cộng vào loss
TRAIN_DECODER_INTERCEPT = True     # Dynamic Logits - giữ nguyên
USE_CONTRASTIVE_LOSS = True        # giữ nguyên
EPOCHS = 5                         # theo yêu cầu (bản N006/N009 gốc dùng 44)
```

L_total = L_CE + `0 * L_hall` + gate(epoch) · lambda_con · L_con, cộng với Dynamic Decoder
Intercept vẫn hạ logits object vắng mặt trực tiếp ở đầu ra decoder (độc lập với loss).

## Vì sao không chỉ tắt object_token_ids

`hallucination_object_loss` (HalluLoss) và `apply_decoder_object_intercept` (Dynamic Logits)
dùng CHUNG hạ tầng `object_token_ids`/COCO-instances ground truth với `build_negative_captions`
(Contrastive). Nếu tắt hạ tầng này theo `USE_HALL_LOSS` như bản gốc, Dynamic Logits và
Contrastive cũng sẽ vô tác dụng theo (silent no-op). Notebook này tách riêng bằng cờ
`USE_OBJECT_GROUNDING = USE_HALL_LOSS or TRAIN_DECODER_INTERCEPT or USE_CONTRASTIVE_LOSS`: hạ
tầng vẫn được dựng khi Dynamic Logits/Contrastive cần, còn `current_lambda_hall` bị ép về `0.0`
khi `USE_HALL_LOSS=False` — `hall_loss` vẫn được tính và log để tham khảo, chỉ không cộng vào
loss dùng để backward.

## Trạng thái

Notebook này **được push lên Kaggle để chạy thật với GPU** (khác với N009 - chỉ push để lưu
trữ/soát code, không chạy). Xem `research/notebooks/manifest.csv` (`N010`).


In [ ]:
import os
import math
import re
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from collections import Counter
from transformers import CLIPVisionModel, get_linear_schedule_with_warmup
import json  # Thêm json để xử lý COCO annotation và lưu vocab

# ==========================================
# CẤU HÌNH VÀ SIÊU THAM SỐ (TRAIN FROM SCRATCH)
# ==========================================
SEED = 42

def seed_everything(seed=SEED):
    """Cố định RNG để các lần train có thể tái lập."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    """Seed Python/NumPy trong từng DataLoader worker."""
    worker_seed = torch.initial_seed() % (2 ** 32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Đường dẫn COCO 2014
TRAIN_IMG_DIR = "/kaggle/input/datasets/nadaibrahim/coco2014/train2014/train2014"
VAL_IMG_DIR = "/kaggle/input/datasets/nadaibrahim/coco2014/val2014/val2014"
KARPATHY_JSON_FILE = "/kaggle/input/datasets/nganhthnh/karpathy-split/dataset_coco.json"
TRAIN_INSTANCES_JSON = "/kaggle/input/datasets/nadaibrahim/coco2014/captions/annotations/instances_train2014.json"
VAL_INSTANCES_JSON = "/kaggle/input/datasets/nadaibrahim/coco2014/captions/annotations/instances_val2014.json"

# Đường dẫn lưu trữ - đặt tên riêng "scratch" để không đè lên checkpoint của bản fine-tune cũ
OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

VOCAB_FILE = os.path.join(OUTPUT_DIR, "coco2014_karpathy_dynlogits_contrastive_vocab.json")
CHECKPOINT_FILE = os.path.join(OUTPUT_DIR, "coco2014_karpathy_dynlogits_contrastive_last_checkpoint.pth")
BEST_MODEL_FILE = os.path.join(OUTPUT_DIR, "coco2014_karpathy_dynlogits_contrastive_best_model.pth")

# Ablation MỚI (thêm contrastive loss + projection head mới) -> train từ đầu, KHÔNG kế thừa
# checkpoint của cấu hình dynamic-logits+fixloss (kiến trúc đã đổi, load lại sẽ lệch shape).
# Nếu kernel bị Kaggle ngắt giữa chừng, đổi RESUME_TRAINING=True và trỏ RESUME_CHECKPOINT_FILE
# về đúng last_checkpoint mà CHÍNH bản contrastive này đã lưu (không phải checkpoint bản khác).
RESUME_TRAINING = False
RESUME_CHECKPOINT_FILE = CHECKPOINT_FILE
FINE_TUNE_RESET_TRAINING_STATE = False

# Hallucination-aware training: dùng COCO instances (ground truth) để phạt object vắng mặt.
# Không dùng YOLO nữa (bỏ hẳn fallback detector) - ảnh không có annotation COCO instances thì đơn
# giản là bỏ qua penalty cho ảnh đó, thay vì đoán bằng object detector kém tin cậy hơn.
USE_HALL_LOSS = False  # tắt theo yêu cầu: chỉ dùng Dynamic Logits (decoder intercept) + Contrastive
USE_COCO_INSTANCE_HALL_LABELS = True
USE_YOLO_HALL_LOSS = False
LAMBDA_HALL = 0.01
HALL_LAMBDA_WARMUP_EPOCHS = 3
HALL_LOSS_MIN_STEP_RATIO = 0.3
HALL_DEBUG_BATCHES = 2

# --- Contrastive loss ảnh-caption (xem research/contrastive-loss-for-caption-hallucination.md) ---
# Positive = caption đúng (teacher forcing); negative = caption đúng nhưng bị thay 1 object đang
# CÓ trong ảnh bằng 1 object KHÔNG có trong ảnh (mục 5.1 tài liệu) -> tái dùng đúng
# COCO_OBJECT_SYNONYMS/object_token_ids đã có sẵn cho HalluLoss, không cần nguồn dữ liệu mới.
USE_CONTRASTIVE_LOSS = True
LAMBDA_CONTRASTIVE = 0.01  # nâng tu 0.005 (mục 8 tai lieu: "neu CHAIR khong giam du, thu lambda_con=0.01")
CONTRASTIVE_TAU = 0.07
CONTRASTIVE_WARMUP_EPOCHS = 3  # gate(epoch) = min(1, epoch/warmup), giống cách HALL_LAMBDA_WARMUP_EPOCHS
CONTRASTIVE_PROJ_DIM = 256  # chiều chung để chiếu image/text embedding trước khi tính cosine similarity
USE_AMP = True  # mixed precision (fp16 autocast + GradScaler) cho vong train - chi tang toc, KHONG dung cho vong val de giu metric val nguyen fp32 nhu cac run truoc

IMAGE_SIZE = (224, 224)
SEQ_LENGTH = 25
MIN_WORD_FREQ = 5
EMBED_DIM = 512
# --- Giữ NGUYÊN kiến trúc decoder giống hệt bản baseline/fine-tune (FF_DIM=512, NUM_LAYERS=2) ---
# Để so sánh baseline vs bản train-from-scratch (hallucination loss + decoder intercept + BLEU
# checkpoint selection) là fair, KHÔNG đổi kiến trúc ở đây - nếu đổi cùng lúc, kết quả tốt hơn sẽ
# không biết là do decoder to hơn hay do các cải tiến hallucination đang muốn đánh giá. Muốn thử
# kiến trúc lớn hơn thì đổi trực tiếp 2 số này, coi đó là 1 thí nghiệm RIÊNG, KHÔNG gộp chung
# với thí nghiệm so sánh baseline.
FF_DIM = 512
NUM_LAYERS = 2
NUM_HEADS = 8
BATCH_SIZE = 32

# Dùng lại đúng LR baseline-from-scratch của bạn (1e-4) để so sánh fair - không đoán 1 giá trị
# mới (vd 4e-4) khi chưa có bằng chứng nó hợp với đúng kiến trúc/data pipeline này. Fine-tune
# (tiếp tục train từ checkpoint có sẵn) mới cần LR nhỏ hơn hẳn (vd 1e-5), không áp dụng ở đây.
EPOCHS = 5  # theo yêu cầu: chỉ train 5 epoch cho ablation này (không phải 44 như bản N006/N009)
LEARNING_RATE = 1e-4

# --- Cải tiến chống overfit / hallucination khi train from scratch ---
LABEL_SMOOTHING = 0.1  # giảm overconfidence -> giảm hallucination
GRAD_CLIP_NORM = 1.0   # ổn định gradient khi train decoder từ đầu

# --- Dynamic intercept ngay tại đầu ra decoder TRONG LÚC TRAIN ---
# Bản cũ dùng penalty tĩnh cho mọi timestep nên dễ làm méo CE loss và có thể khiến CHAIR tăng.
# Bản dynamic này mềm hơn:
# 1) ramp-up penalty theo epoch để các epoch đầu học ngôn ngữ/grounding ổn trước,
# 2) chỉ phạt mạnh ở timestep mà model đang gán xác suất cao cho nhóm object vắng mặt,
# 3) không phạt token target đúng khi tính CE, tránh xung đột nếu COCO instances thiếu object.
TRAIN_DECODER_INTERCEPT = True
TRAIN_INTERCEPT_MODE = "dynamic"
TRAIN_INTERCEPT_MAX_PENALTY = 0.5
TRAIN_INTERCEPT_WARMUP_EPOCHS = 5
TRAIN_INTERCEPT_PROB_THRESHOLD = 0.08
TRAIN_INTERCEPT_TARGET_PROTECT = True

# --- Cờ tổng hợp: có cần dựng object_token_ids/COCO-instance-objects hay không? ---
# 3 thành phần dùng CHUNG hạ tầng grounding này: HalluLoss (phạt loss), Dynamic Decoder
# Intercept (hạ logits) và Contrastive loss (tạo negative caption). USE_HALL_LOSS=False KHÔNG
# được phép tắt luôn hạ tầng này nếu Dynamic Intercept hoặc Contrastive vẫn cần nó - nếu không,
# apply_decoder_object_intercept/build_negative_captions sẽ luôn no-op dù bật cờ tương ứng.
# USE_HALL_LOSS chỉ còn quyết định lambda_hall có được cộng vào loss hay không (xem get_hall_lambda
# ở dưới, đã đổi thành current_lambda_hall = 0.0 khi USE_HALL_LOSS=False).
USE_OBJECT_GROUNDING = USE_HALL_LOSS or TRAIN_DECODER_INTERCEPT or USE_CONTRASTIVE_LOSS

# --- Theo dõi / early-stop bằng metric caption thật (BLEU-4) thay vì chỉ val_loss ---
# val_loss thấp không đồng nghĩa caption tốt hơn: dùng BLEU-4 chuẩn COCO caption (pycocoevalcap,
# KHÔNG dùng nltk vì nltk BLEU vốn thiết kế cho dịch máy, không khớp cách đánh giá image
# captioning trong các paper/benchmark COCO) trên 1 subset ảnh val mỗi epoch (greedy decode)
# để chọn best model theo chất lượng caption thực tế.
EARLY_STOP_METRIC = "loss"  # "bleu4" | "loss"
VAL_METRIC_MAX_SAMPLES = 1000

PAD_IDX = 0
START_IDX = 1
END_IDX = 2
UNK_IDX = 3

# ==========================================
# TIỀN XỬ LÝ DỮ LIỆU
# ==========================================
def custom_standardization(text):
    text = text.lower()
    strip_chars = r"""!"#$%&'()*+,-./:;=?@[\]^_`{|}~1234567890"""
    text = re.sub(f"[{re.escape(strip_chars)}]", "", text)
    return text.strip()

# Train: Có Augmentation (giảm RandomRotation so với bản fine-tune: 36 -> 15 độ, vì xoay mạnh
# làm lệch phân bố ảnh so với lúc CLIP pretrain và không cập nhật lại caption theo hướng xoay
# -> nhiễu label, đặc biệt hại khi train from scratch).
train_transforms = T.Compose([
    T.Resize(IMAGE_SIZE),
    T.CenterCrop(IMAGE_SIZE),
    T.RandomHorizontalFlip(),
    T.RandomRotation(degrees=15),
    T.ColorJitter(contrast=0.3),
    T.ToTensor(),
    T.Normalize(mean=[0.48145466, 0.4578275, 0.40821073],
                std=[0.26862954, 0.26130258, 0.27577711])
])

# Validation: KHÔNG Augmentation
val_transforms = T.Compose([
    T.Resize(IMAGE_SIZE),
    T.CenterCrop(IMAGE_SIZE),
    T.ToTensor(),
    T.Normalize(mean=[0.48145466, 0.4578275, 0.40821073],
                std=[0.26862954, 0.26130258, 0.27577711])
])

class COCODataset(Dataset):
    def __init__(self, image_paths, captions, vocab, transforms=None, image_objects=None):
        self.image_paths = image_paths
        self.captions = captions
        self.vocab = vocab
        self.transforms = transforms
        self.image_objects = image_objects or {}

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        caption = self.captions[idx]
        gt_objects_text = "|".join(sorted(self.image_objects.get(os.path.basename(img_path), set())))

        try:
            image = Image.open(img_path).convert("RGB")
        except Exception as e:
            # Fallback nếu ảnh bị lỗi (Rất hiếm nhưng có thể xảy ra ở bộ dữ liệu lớn)
            image = Image.new("RGB", IMAGE_SIZE, (0, 0, 0))

        if self.transforms:
            image = self.transforms(image)

        tokens = [self.vocab.get(word, UNK_IDX) for word in caption.split()]
        tokens = [START_IDX] + tokens[:SEQ_LENGTH-2] + [END_IDX]

        padded_tokens = tokens + [PAD_IDX] * (SEQ_LENGTH - len(tokens))
        target_tensor = torch.tensor(padded_tokens, dtype=torch.long)

        return image, target_tensor, img_path, gt_objects_text

def load_karpathy_data(json_file, split="train"):
    """Đọc COCO Karpathy split từ dataset_coco.json."""
    print(f"Đang đọc Karpathy split '{split}' từ: {json_file}...")
    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    image_paths = []
    captions = []
    missing_images = 0

    for img in data.get('images', []):
        if img.get('split') != split:
            continue

        filepath = img.get('filepath', '')
        filename = img.get('filename')
        if not filename:
            continue

        image_dir = TRAIN_IMG_DIR if filepath == 'train2014' else VAL_IMG_DIR
        candidates = [
            os.path.join(image_dir, filename),
            os.path.join(os.path.dirname(image_dir), filename),
            os.path.join(os.path.dirname(image_dir), filepath, filename),
        ]
        img_path = next((path for path in candidates if os.path.exists(path)), None)
        if img_path is None:
            missing_images += 1
            continue

        for sent in img.get('sentences', []):
            if 'raw' in sent:
                caption = custom_standardization(sent['raw'])
            else:
                caption = custom_standardization(' '.join(sent.get('tokens', [])))

            if caption:
                image_paths.append(img_path)
                captions.append(caption)

    print(f"-> Karpathy {split}: {len(captions)} cặp ảnh-caption.")
    if missing_images:
        print(f"-> Bỏ qua {missing_images} ảnh vì không tìm thấy file ảnh.")
    return image_paths, captions

def load_coco_data(annotation_file, image_dir):
    """Hàm đọc dữ liệu JSON của COCO Dataset"""
    print(f"Đang đọc dữ liệu từ: {annotation_file}...")
    with open(annotation_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # Tạo mapping từ ID ảnh sang Tên file
    img_id_to_filename = {img['id']: img['file_name'] for img in data['images']}

    image_paths = []
    captions = []

    for ann in data['annotations']:
        img_id = ann['image_id']
        caption = custom_standardization(ann['caption'])

        if img_id in img_id_to_filename:
            filename = img_id_to_filename[img_id]
            img_path = os.path.join(image_dir, filename)

            # Chỉ thêm nếu file thực sự tồn tại
            if os.path.exists(img_path):
                image_paths.append(img_path)
                captions.append(caption)

    print(f"-> Đã tải {len(captions)} cặp ảnh-caption.")
    return image_paths, captions

def build_or_load_vocab(captions, min_freq=5, vocab_file=VOCAB_FILE):
    """Hàm tạo Vocabulary mới hoặc tải từ file nếu đã tồn tại"""
    if os.path.exists(vocab_file):
        print(f"-> Tìm thấy file Vocab. Đang tải Vocabulary từ {vocab_file}...")
        with open(vocab_file, 'r', encoding='utf-8') as f:
            vocab = json.load(f)
        print(f"-> Kích thước Vocabulary đã tải: {len(vocab)}")
        return vocab

    print(f"-> Không tìm thấy file {vocab_file}. Đang xây dựng Vocabulary mới...")
    all_words = []
    for cap in captions:
        all_words.extend(cap.split())

    word_counts = Counter(all_words)
    valid_words = [word for word, count in word_counts.items() if count >= min_freq]

    vocab = {"<pad>": PAD_IDX, "<start>": START_IDX, "<end>": END_IDX, "<unk>": UNK_IDX}
    for idx, word in enumerate(valid_words):
        vocab[word] = idx + 4

    # Lưu lại Vocab để tái sử dụng
    with open(vocab_file, 'w', encoding='utf-8') as f:
        json.dump(vocab, f, ensure_ascii=False)

    print(f"-> Kích thước Vocabulary (từ xuất hiện >= {min_freq} lần): {len(vocab)}")
    print(f"-> Đã lưu Vocabulary tại: {vocab_file}")
    return vocab

In [ ]:
# ==========================================
# KIẾN TRÚC MÔ HÌNH
# ==========================================

# ==========================================
# HALLUCINATION-AWARE LOSS DỰA TRÊN COCO INSTANCES (GROUND TRUTH)
# ==========================================
COCO_OBJECT_SYNONYMS = {
    "person": ["person", "people", "man", "woman", "boy", "girl", "child", "children"],
    "bicycle": ["bicycle", "bike"], "car": ["car", "auto", "automobile"],
    "motorcycle": ["motorcycle", "motorbike"], "airplane": ["airplane", "plane", "aircraft"],
    "bus": ["bus"], "train": ["train"], "truck": ["truck"], "boat": ["boat"],
    "traffic light": ["traffic", "light"], "fire hydrant": ["hydrant"], "stop sign": ["stop", "sign"],
    "parking meter": ["meter"], "bench": ["bench"], "bird": ["bird"], "cat": ["cat"], "dog": ["dog"],
    "horse": ["horse"], "sheep": ["sheep"], "cow": ["cow"], "elephant": ["elephant"], "bear": ["bear"],
    "zebra": ["zebra"], "giraffe": ["giraffe"], "backpack": ["backpack"], "umbrella": ["umbrella"],
    "handbag": ["handbag", "bag", "purse"], "tie": ["tie"], "suitcase": ["suitcase", "luggage"],
    "frisbee": ["frisbee"], "skis": ["skis"], "snowboard": ["snowboard"], "sports ball": ["ball"],
    "kite": ["kite"], "baseball bat": ["bat"], "baseball glove": ["glove"], "skateboard": ["skateboard"],
    "surfboard": ["surfboard"], "tennis racket": ["racket"], "bottle": ["bottle"], "wine glass": ["glass"],
    "cup": ["cup"], "fork": ["fork"], "knife": ["knife"], "spoon": ["spoon"], "bowl": ["bowl"],
    "banana": ["banana"], "apple": ["apple"], "sandwich": ["sandwich"], "orange": ["orange"],
    "broccoli": ["broccoli"], "carrot": ["carrot"], "hot dog": ["hotdog"], "pizza": ["pizza"],
    "donut": ["donut", "doughnut"], "cake": ["cake"], "chair": ["chair"], "couch": ["couch", "sofa"],
    "potted plant": ["plant"], "bed": ["bed"], "dining table": ["table"], "toilet": ["toilet"],
    "tv": ["tv", "television"], "laptop": ["laptop"], "mouse": ["mouse"], "remote": ["remote"],
    "keyboard": ["keyboard"], "cell phone": ["phone", "cellphone"], "microwave": ["microwave"],
    "oven": ["oven"], "toaster": ["toaster"], "sink": ["sink"], "refrigerator": ["refrigerator", "fridge"],
    "book": ["book"], "clock": ["clock"], "vase": ["vase"], "scissors": ["scissors"],
    "teddy bear": ["teddy"], "hair drier": ["drier", "dryer"], "toothbrush": ["toothbrush"],
}

def build_object_token_ids(vocab):
    object_token_ids = {}
    for obj_name, synonyms in COCO_OBJECT_SYNONYMS.items():
        ids = sorted({vocab[word] for word in synonyms if word in vocab})
        if ids:
            object_token_ids[obj_name] = ids
    print(f"-> Hall loss theo dõi {len(object_token_ids)} object có token trong vocabulary.")
    return object_token_ids


def load_coco_instance_objects(instances_file):
    if not instances_file or not os.path.exists(instances_file):
        print(f"[!] Không tìm thấy COCO instances: {instances_file}. Hallucination loss sẽ bỏ qua các ảnh không có annotation.")
        return {}

    print(f"-> Đang đọc object ground-truth từ: {instances_file}")
    with open(instances_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    cat_id_to_name = {cat["id"]: cat["name"] for cat in data.get("categories", [])}
    image_id_to_filename = {img["id"]: img["file_name"] for img in data.get("images", [])}
    image_objects = {}

    for ann in data.get("annotations", []):
        filename = image_id_to_filename.get(ann.get("image_id"))
        obj_name = cat_id_to_name.get(ann.get("category_id"))
        if filename and obj_name in COCO_OBJECT_SYNONYMS:
            image_objects.setdefault(filename, set()).add(obj_name)

    print(f"-> Đã nạp object ground-truth cho {len(image_objects)} ảnh từ COCO instances.")
    return image_objects

HALL_DEBUG_STATE = {"calls": 0}

def hallucination_object_loss(
    outputs,
    image_paths,
    gt_objects_batch,
    object_token_ids,
    token_mask=None,
    min_step_index=0,
):
    """Object-level unlikelihood loss đã làm mềm để phối hợp với dynamic logits.

    - Dùng mean trên nhóm token absent thay vì sum để ảnh có nhiều object vắng mặt
      không bị phạt mạnh chỉ vì số lượng token cần xét lớn hơn.
    - Chỉ phạt từ min_step_index trở đi để tránh nhiễu ở đầu câu, khi decoder còn
      ít ngữ cảnh ngôn ngữ.
    """
    if not object_token_ids:
        return outputs.new_tensor(0.0)

    probs = outputs.softmax(dim=-1)
    seq_len = outputs.size(1)
    if min_step_index > 0:
        step_positions = torch.arange(seq_len, device=outputs.device)
        step_position_mask = (step_positions >= min_step_index).float()
    else:
        step_position_mask = None

    losses = []
    num_with_gt_objects = 0
    num_with_present_objects = 0
    num_with_absent_tokens = 0
    for batch_index, image_path in enumerate(image_paths):
        gt_text = gt_objects_batch[batch_index] if gt_objects_batch is not None else ""
        gt_objects = set(gt_text.split("|")) if gt_text else set()
        if gt_objects:
            present_objects = gt_objects
            num_with_gt_objects += 1
            num_with_present_objects += 1
        else:
            continue

        absent_token_ids = []
        for obj_name, token_ids in object_token_ids.items():
            if obj_name not in present_objects:
                absent_token_ids.extend(token_ids)

        if not absent_token_ids:
            continue
        num_with_absent_tokens += 1

        absent_ids = torch.tensor(sorted(set(absent_token_ids)), device=outputs.device, dtype=torch.long)
        absent_probs = probs[batch_index].index_select(dim=-1, index=absent_ids).clamp(max=1.0 - 1e-6)
        step_penalty = -torch.log1p(-absent_probs).mean(dim=-1)

        if token_mask is not None:
            valid_steps = token_mask[batch_index].float()
        else:
            valid_steps = outputs.new_ones(step_penalty.shape)

        if step_position_mask is not None:
            valid_steps = valid_steps * step_position_mask

        valid_steps_sum = valid_steps.sum()
        if valid_steps_sum.item() <= 0:
            continue

        losses.append((step_penalty * valid_steps).sum() / valid_steps_sum.clamp_min(1.0))

    if not losses:
        if HALL_DEBUG_STATE["calls"] < HALL_DEBUG_BATCHES:
            print(
                f"[HallDebug] batch_size={len(image_paths)} | object_groups={len(object_token_ids)} | "
                f"gt_images={num_with_gt_objects} | present_images={num_with_present_objects} | "
                f"penalized_images={num_with_absent_tokens} | hall_ul=0.00000000"
            )
            HALL_DEBUG_STATE["calls"] += 1
        return outputs.new_tensor(0.0)

    if HALL_DEBUG_STATE["calls"] < HALL_DEBUG_BATCHES:
        debug_value = torch.stack(losses).mean().detach().item()
        print(
            f"[HallDebug] batch_size={len(image_paths)} | object_groups={len(object_token_ids)} | "
            f"gt_images={num_with_gt_objects} | present_images={num_with_present_objects} | "
            f"penalized_images={num_with_absent_tokens} | hall_ul={debug_value:.8f}"
        )
        HALL_DEBUG_STATE["calls"] += 1
    return torch.stack(losses).mean()


def get_hall_lambda(epoch, max_lambda=LAMBDA_HALL, warmup_epochs=HALL_LAMBDA_WARMUP_EPOCHS):
    if warmup_epochs <= 0:
        return max_lambda
    progress = min(1.0, float(epoch + 1) / float(warmup_epochs))
    return max_lambda * progress


# ==========================================
# CONTRASTIVE LOSS ẢNH-CAPTION (research/contrastive-loss-for-caption-hallucination.md)
# ==========================================
def build_object_id_to_name(object_token_ids):
    """Đảo ngược object_token_ids (obj_name -> [token_id,...]) đã có sẵn cho HalluLoss thành
    token_id -> obj_name, dùng để dò object-token trong 1 caption khi tạo negative."""
    id_to_name = {}
    for obj_name, token_ids in object_token_ids.items():
        for token_id in token_ids:
            id_to_name[token_id] = obj_name
    return id_to_name


def build_negative_captions(captions, gt_objects_batch, object_token_ids, id_to_obj_name):
    """Tạo negative caption cho contrastive loss (mục 5.1/5.2 tài liệu nghiên cứu).

    Cách 1 (ưu tiên, mục 5.1 - "negative bằng thay object"): tìm 1 vị trí token trong caption
    thuộc object ĐANG CÓ thật trong ảnh (theo COCO instances ground truth), thay bằng token của
    1 object KHÔNG có trong ảnh -> negative chắc chắn là hallucination thật, không đoán mò.

    Cách 2 (fallback, mục 5.2 - "negative bằng thêm object vắng mặt"): nếu caption không chứa
    token object nào nhận diện được (hoặc ảnh không có object nào để thay), chèn 1 object vắng
    mặt vào ngay trước token cuối cùng có nghĩa (trước <end>/<pad> đầu tiên).

    Nếu ảnh không có object nào trong COCO_OBJECT_SYNONYMS coi là vắng mặt (hiếm), giữ nguyên
    caption -> loss cho mẫu đó gần 0, không crash.
    """
    negative_captions = captions.clone()
    all_object_names = list(object_token_ids.keys())

    for batch_index in range(captions.size(0)):
        gt_text = gt_objects_batch[batch_index] if gt_objects_batch is not None else ""
        present_objects = set(gt_text.split("|")) if gt_text else set()
        absent_names = [name for name in all_object_names if name not in present_objects]
        if not absent_names:
            continue

        replacement_name = random.choice(absent_names)
        replacement_id = object_token_ids[replacement_name][0]

        seq = captions[batch_index].tolist()
        swap_pos = next(
            (pos for pos, tok in enumerate(seq) if id_to_obj_name.get(tok) in present_objects),
            None,
        )

        if swap_pos is not None:
            negative_captions[batch_index, swap_pos] = replacement_id
            continue

        # Fallback: không có object-token nào để thay -> chèn object vắng mặt trước <end>/<pad>
        # đầu tiên (vị trí > 1 để không đè lên <start>).
        insert_pos = next((pos for pos, tok in enumerate(seq) if tok in (END_IDX, PAD_IDX)), None)
        if insert_pos is not None and insert_pos > 1:
            negative_captions[batch_index, insert_pos - 1] = replacement_id
        # Nếu caption quá ngắn để chèn an toàn, giữ nguyên (negative == positive, loss ~ 0).

    return negative_captions


def get_contrastive_lambda(epoch, max_lambda=LAMBDA_CONTRASTIVE, warmup_epochs=CONTRASTIVE_WARMUP_EPOCHS):
    """gate(epoch) * lambda_con, cùng công thức warmup với get_hall_lambda (mục 6.3 tài liệu)."""
    if warmup_epochs <= 0:
        return max_lambda
    progress = min(1.0, float(epoch + 1) / float(warmup_epochs))
    return max_lambda * progress


def contrastive_image_caption_loss(image_embed, pos_text_embed, neg_text_embed, tau=CONTRASTIVE_TAU):
    """InfoNCE tối giản với ĐÚNG 1 negative/ảnh (mục 6.1 + khuyến nghị "bắt đầu đơn giản" ở mục 8
    của tài liệu nghiên cứu): softmax trên {positive, negative}, label luôn là vị trí positive."""
    pos_sim = (image_embed * pos_text_embed).sum(dim=-1) / tau
    neg_sim = (image_embed * neg_text_embed).sum(dim=-1) / tau
    logits = torch.stack([pos_sim, neg_sim], dim=1)  # (B, 2) - cot 0 la positive
    labels = torch.zeros(logits.size(0), dtype=torch.long, device=logits.device)
    return nn.functional.cross_entropy(logits, labels)


def _intercept_epoch_scale(epoch_index):
    if TRAIN_INTERCEPT_WARMUP_EPOCHS <= 0:
        return 1.0
    return min(1.0, float(epoch_index + 1) / float(TRAIN_INTERCEPT_WARMUP_EPOCHS))


def apply_decoder_object_intercept(
    outputs,
    image_paths,
    gt_objects_batch,
    object_token_ids,
    targets=None,
    epoch_index=0,
):
    """Dynamic logit adjustment cho token thuộc object vắng mặt.

    Khác bản static cũ, penalty không còn là một hằng số áp cho mọi timestep. Với mỗi ảnh:
    - xác định các object không có trong COCO instances,
    - đo tổng xác suất hiện tại mà model gán cho các token object vắng mặt ở từng timestep,
    - chỉ hạ logits mạnh khi xác suất nhóm này vượt ngưỡng,
    - bảo vệ token target đúng để CE không bị phạt ngược nếu annotation thiếu object.
    """
    if (
        not TRAIN_DECODER_INTERCEPT
        or TRAIN_INTERCEPT_MODE == "off"
        or not object_token_ids
    ):
        return outputs

    epoch_scale = _intercept_epoch_scale(epoch_index)
    if epoch_scale <= 0:
        return outputs

    full_bias = torch.zeros_like(outputs)
    probs = outputs.detach().softmax(dim=-1)
    max_penalty = TRAIN_INTERCEPT_MAX_PENALTY * epoch_scale

    for batch_index, image_path in enumerate(image_paths):
        gt_text = gt_objects_batch[batch_index] if gt_objects_batch is not None else ""
        present_objects = set(gt_text.split("|")) if gt_text else set()
        if not present_objects:
            continue

        absent_ids = [
            token_id
            for obj_name, token_ids in object_token_ids.items()
            if obj_name not in present_objects
            for token_id in token_ids
        ]
        if not absent_ids:
            continue

        absent_ids_t = torch.tensor(sorted(set(absent_ids)), device=outputs.device, dtype=torch.long)

        if TRAIN_INTERCEPT_MODE == "static":
            step_penalty = outputs.new_full((outputs.size(1),), max_penalty)
        else:
            absent_mass = probs[batch_index].index_select(dim=-1, index=absent_ids_t).sum(dim=-1)
            normalized = (
                (absent_mass - TRAIN_INTERCEPT_PROB_THRESHOLD)
                / max(1e-6, 1.0 - TRAIN_INTERCEPT_PROB_THRESHOLD)
            ).clamp(min=0.0, max=1.0)
            step_penalty = max_penalty * normalized

        full_bias[batch_index, :, absent_ids_t] = -step_penalty.unsqueeze(-1)

        if TRAIN_INTERCEPT_TARGET_PROTECT and targets is not None:
            valid = targets[batch_index] != PAD_IDX
            timesteps = torch.arange(outputs.size(1), device=outputs.device)[valid]
            target_ids = targets[batch_index, valid]
            full_bias[batch_index, timesteps, target_ids] = 0.0

    return outputs + full_bias

class CLIPEncoder(nn.Module):
    def __init__(self, embed_dim):
        super(CLIPEncoder, self).__init__()
        self.clip_vision = CLIPVisionModel.from_pretrained(
            "openai/clip-vit-base-patch16",
            use_safetensors=True,
        )

        for param in self.clip_vision.parameters():
            param.requires_grad = False

        self.fc = nn.Linear(768, embed_dim)
        self.layer_norm = nn.LayerNorm(embed_dim)

    def forward(self, images):
        outputs = self.clip_vision(pixel_values=images)
        features = outputs.last_hidden_state
        embeddings = self.fc(features)
        return self.layer_norm(embeddings)

class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, dropout=0.1, max_len=100):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, embed_dim)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, embed_dim, 2).float() * (-math.log(10000.0) / embed_dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class ImageCaptioningTransformer(nn.Module):
    def __init__(self, vocab_size, embed_dim, ff_dim, num_heads, num_layers, contrastive_proj_dim=CONTRASTIVE_PROJ_DIM):
        super(ImageCaptioningTransformer, self).__init__()
        self.encoder = CLIPEncoder(embed_dim)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.pos_encoder = PositionalEncoding(embed_dim)

        decoder_layer = nn.TransformerDecoderLayer(d_model=embed_dim, nhead=num_heads, dim_feedforward=ff_dim, batch_first=True)
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.fc_out = nn.Linear(embed_dim, vocab_size, bias=False)
        self.fc_out.weight = self.embedding.weight

        # --- Projection head cho contrastive loss (mới so với N006) ---
        # Tham số MỚI, không có trong checkpoint cũ -> load_checkpoint() phải dùng strict=False.
        self.image_proj = nn.Linear(embed_dim, contrastive_proj_dim)
        self.text_proj = nn.Linear(embed_dim, contrastive_proj_dim)

    def generate_square_subsequent_mask(self, sz):
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask

    def encode_image(self, images):
        """Tách riêng bước encode ảnh để có thể dùng chung 1 lần `memory` cho cả forward pass
        positive VÀ negative caption trong contrastive loss (tránh forward lại CLIP-ViT 2 lần)."""
        return self.encoder(images)

    def decode(self, memory, captions, return_hidden=False):
        """Tách riêng bước decode để contrastive loss lấy được decoder hidden state (trước
        fc_out) làm text representation, không chỉ logits."""
        tgt_embed = self.embedding(captions)
        tgt_embed = self.pos_encoder(tgt_embed)
        tgt_mask = self.generate_square_subsequent_mask(captions.size(1)).to(captions.device)
        tgt_key_padding_mask = (captions == PAD_IDX)
        hidden = self.transformer_decoder(tgt=tgt_embed, memory=memory,
                                          tgt_mask=tgt_mask,
                                          tgt_key_padding_mask=tgt_key_padding_mask)
        logits = self.fc_out(hidden)
        if return_hidden:
            return logits, hidden
        return logits

    def forward(self, images, captions, return_hidden=False):
        memory = self.encode_image(images)
        return self.decode(memory, captions, return_hidden=return_hidden)

    def project_image(self, memory):
        """v = normalize(W_v * mean(image_features)) - đúng công thức mục 7 tài liệu nghiên cứu."""
        pooled = memory.mean(dim=1)
        return nn.functional.normalize(self.image_proj(pooled), dim=-1)

    def project_text(self, hidden, padding_mask):
        """t = normalize(W_t * mean(decoder_hidden_states)), chỉ mean-pool trên vị trí KHÔNG phải
        pad (padding_mask=True tại vị trí pad, giống tgt_key_padding_mask)."""
        valid = (~padding_mask).unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * valid).sum(dim=1) / valid.sum(dim=1).clamp_min(1.0)
        return nn.functional.normalize(self.text_proj(pooled), dim=-1)

In [ ]:
def _unwrap_model(model):
    return model.module if isinstance(model, nn.DataParallel) else model

def _prepare_save_path(filepath):
    """Đảm bảo checkpoint được lưu vào thư mục ghi được, không phải /kaggle/input."""
    if os.path.isdir(filepath):
        raise RuntimeError(
            f"'{filepath}' là thư mục, không phải file checkpoint. "
            "Hãy truyền CHECKPOINT_FILE hoặc BEST_MODEL_FILE, ví dụ /kaggle/working/last_checkpoint.pth."
        )
    save_dir = os.path.dirname(filepath) or "."
    os.makedirs(save_dir, exist_ok=True)
    if not os.access(save_dir, os.W_OK):
        raise RuntimeError(
            f"Không thể ghi checkpoint vào '{save_dir}'. "
            "Trên Kaggle, hãy lưu vào /kaggle/working và chỉ load checkpoint từ /kaggle/input."
        )
    return filepath

def _resolve_checkpoint_path(filepath):
    """Cho phép RESUME_CHECKPOINT_FILE là file .pth hoặc folder chứa checkpoint."""
    if os.path.isfile(filepath):
        return filepath

    if os.path.isdir(filepath):
        preferred_names = ["last_checkpoint.pth", "best_model.pth"]
        for name in preferred_names:
            candidate = os.path.join(filepath, name)
            if os.path.isfile(candidate):
                return candidate

        checkpoint_files = []
        for root, _, files in os.walk(filepath):
            for name in files:
                if name.endswith((".pth", ".pt")):
                    checkpoint_files.append(os.path.join(root, name))

        if checkpoint_files:
            checkpoint_files.sort(key=os.path.getmtime, reverse=True)
            return checkpoint_files[0]

        raise RuntimeError(f"'{filepath}' là thư mục nhưng không tìm thấy file .pth/.pt bên trong.")

    return filepath

class EarlyStoppingCallback:
    """So với bản fine-tune: hỗ trợ mode="max" để early-stop/chọn best model theo BLEU-4
    (điểm càng cao càng tốt) thay vì chỉ mode="min" theo val_loss.

    patience mặc định 5 (thay vì 3): BLEU-4 tính bằng greedy decode trên subset mỗi epoch nhiễu
    hơn nhiều so với val_loss (quá trình rời rạc, không mượt/đơn điệu như loss) - patience thấp dễ
    dừng sớm giữa 1 đợt dao động tạm thời, mất cơ hội cải thiện tiếp."""
    def __init__(self, patience=5, min_delta=0.0, save_path=BEST_MODEL_FILE, mode="min"):
        assert mode in ("min", "max")
        self.patience = patience
        self.min_delta = min_delta
        self.save_path = save_path
        self.mode = mode
        self.counter = 0
        self.best_score = float('inf') if mode == "min" else float('-inf')
        self.early_stop = False

    def _improved(self, score):
        if self.mode == "min":
            return score < self.best_score - self.min_delta
        return score > self.best_score + self.min_delta

    def __call__(self, score, model, optimizer, epoch):
        if self._improved(score):
            self.best_score = score
            self.counter = 0
            self.save_best_model(model, optimizer, epoch, score)
        else:
            self.counter += 1
            print(f" EarlyStopping counter: {self.counter} / {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True

    def save_best_model(self, model, optimizer, epoch, score):
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': _unwrap_model(model).state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'seed': SEED,
            'monitor_metric': EARLY_STOP_METRIC,
            'best_score': self.best_score,
        }
        save_path = _prepare_save_path(self.save_path)
        torch.save(checkpoint, save_path)
        print(f" => {EARLY_STOP_METRIC} cải thiện ({score:.4f}). Đã lưu BEST MODEL tại: {save_path}")

# ==========================================
# QUẢN LÝ CHECKPOINT VÀ HUẤN LUYỆN
# ==========================================
def save_checkpoint(model, optimizer, scheduler, epoch, epoch_metrics, early_stopping, filepath=CHECKPOINT_FILE):
    """Lưu Checkpoint bao gồm cả trạng thái Early Stopping và toàn bộ metric của epoch (để vẽ lại sau)."""
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': _unwrap_model(model).state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler is not None else None,
        'seed': SEED,
        'epoch_metrics': epoch_metrics,
        'monitor_metric': EARLY_STOP_METRIC,
        'best_score': early_stopping.best_score,
        'es_counter': early_stopping.counter,
    }
    filepath = _prepare_save_path(filepath)
    torch.save(checkpoint, filepath)
    print(f" => Đã lưu Checkpoint phục hồi tại: {filepath}")

def load_checkpoint(model, optimizer=None, scheduler=None, early_stopping=None, filepath=CHECKPOINT_FILE, resume_training=True, reset_training_state=False):
    """Tải checkpoint; khi fine-tune có thể chỉ load trọng số và reset optimizer/scheduler/epoch."""
    start_epoch = 0
    if not resume_training:
        print("[*] RESUME_TRAINING = False. Sẽ bắt đầu huấn luyện từ Epoch 1.")
        return start_epoch

    filepath = _resolve_checkpoint_path(filepath)

    if os.path.exists(filepath):
        print(f"[*] Tìm thấy Checkpoint '{filepath}'. Đang tải...")
        checkpoint = torch.load(filepath, map_location=device)
        state_dict = checkpoint.get('model_state_dict', checkpoint)
        # strict=False: checkpoint cũ (vd bản không có contrastive loss) không có
        # image_proj/text_proj -> vẫn load được phần trọng số trùng, phần mới khởi tạo lại.
        load_result = _unwrap_model(model).load_state_dict(state_dict, strict=False)
        if load_result.missing_keys or load_result.unexpected_keys:
            print(f"[*] load_state_dict (strict=False): missing_keys={load_result.missing_keys}, "
                  f"unexpected_keys={load_result.unexpected_keys}")

        if reset_training_state:
            start_epoch = 0
            print("[*] Fine-tune mode: chỉ load trọng số model, reset optimizer/scheduler/epoch.")
        else:
            if optimizer is not None and 'optimizer_state_dict' in checkpoint:
                optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            if scheduler is not None and checkpoint.get('scheduler_state_dict') is not None:
                scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
            start_epoch = checkpoint.get('epoch', -1) + 1
            if early_stopping is not None:
                early_stopping.best_score = checkpoint.get('best_score', early_stopping.best_score)
                early_stopping.counter = checkpoint.get('es_counter', 0)

        print(f"[*] Đã khôi phục thành công. Sẽ tiếp tục huấn luyện từ Epoch {start_epoch + 1}")
        if early_stopping is not None:
            print(f"[*] Trạng thái EarlyStopping ({early_stopping.mode}): best_score = {early_stopping.best_score:.4f}, counter = {early_stopping.counter}")
    else:
        print(f"[*] Không tìm thấy Checkpoint '{filepath}'. Sẽ bắt đầu huấn luyện từ Epoch 1.")
    return start_epoch

# ==========================================
# THEO DÕI CHẤT LƯỢNG CAPTION THẬT (BLEU-4, pycocoevalcap) MỖI EPOCH
# ==========================================
def build_val_reference_map(image_paths, captions, max_images=None):
    """Gộp các caption (đã standardize) của cùng 1 ảnh thành list reference string, đúng định
    dạng `gts` mà pycocoevalcap.bleu.Bleu.compute_score() cần (không tokenize thủ công như nltk)."""
    ref_map = {}
    order = []
    for path, cap in zip(image_paths, captions):
        if path not in ref_map:
            ref_map[path] = []
            order.append(path)
        ref_map[path].append(cap)
    if max_images is not None:
        order = order[:max_images]
    return {path: ref_map[path] for path in order}

@torch.no_grad()
def greedy_generate_tokens(model, image_tensor, max_length=SEQ_LENGTH):
    """Sinh caption bằng greedy decoding (nhanh) - CHỈ dùng nội bộ để theo dõi BLEU-4 mỗi epoch
    trong lúc train. Không phải hàm inference cuối cùng - notebook này không có bước generate/
    evaluate caption (xem DECODER_INTERCEPT.md cho constrained decoding lúc inference)."""
    model_ = _unwrap_model(model)
    was_training = model_.training
    model_.eval()

    image_tensor = image_tensor.unsqueeze(0).to(device)
    memory = model_.encoder(image_tensor)
    generated = [START_IDX]

    for _ in range(max_length - 1):
        tgt = torch.tensor(generated, dtype=torch.long, device=device).unsqueeze(0)
        tgt_mask = model_.generate_square_subsequent_mask(tgt.size(1)).to(device)
        tgt_embed = model_.pos_encoder(model_.embedding(tgt))
        out = model_.transformer_decoder(tgt=tgt_embed, memory=memory, tgt_mask=tgt_mask)
        next_token = model_.fc_out(out)[0, -1, :].argmax(dim=-1).item()
        if next_token == END_IDX:
            break
        generated.append(next_token)

    if was_training:
        model_.train()
    return generated[1:]  # bỏ <start>

def evaluate_val_bleu4(model, vocab, ref_map, max_images=VAL_METRIC_MAX_SAMPLES, transforms=None):
    """Tính BLEU-4 bằng pycocoevalcap (chuẩn dùng để báo cáo trong các paper/benchmark image
    captioning trên COCO) trên 1 subset ảnh val bằng greedy decode - KHÔNG dùng nltk vì nltk
    BLEU vốn thiết kế cho đánh giá dịch máy, không phải công cụ chuẩn cho captioning.
    Dùng để theo dõi/early-stop theo chất lượng caption thật thay vì chỉ nhìn val_loss.
    Bỏ qua bước PTBTokenizer (cần Java) vì caption đã được `custom_standardization` chuẩn hoá
    sẵn (viết thường, bỏ dấu câu) - giống cách evaluate_baseline.py trong repo đang dùng."""
    from pycocoevalcap.bleu.bleu import Bleu

    if transforms is None:
        transforms = val_transforms

    idx2word = {idx: word for word, idx in vocab.items()}
    gts, res = {}, {}

    for img_idx, image_path in enumerate(list(ref_map.keys())[:max_images]):
        try:
            image = Image.open(image_path).convert("RGB")
        except Exception:
            continue
        image_tensor = transforms(image)
        token_ids = greedy_generate_tokens(model, image_tensor)
        hyp_caption = " ".join(idx2word.get(tok, "<unk>") for tok in token_ids).strip()

        key = str(img_idx)
        gts[key] = ref_map[image_path]
        res[key] = [hyp_caption if hyp_caption else "<unk>"]

    if not res:
        return 0.0

    bleu_score, _ = Bleu(4).compute_score(gts, res)
    return bleu_score[3]  # BLEU-4

def train_model(train_loader, val_loader, vocab, object_token_ids=None, val_ref_map=None, id_to_obj_name=None):
    vocab_size = len(vocab)
    model = ImageCaptioningTransformer(
        vocab_size=vocab_size, embed_dim=EMBED_DIM,
        ff_dim=FF_DIM, num_heads=NUM_HEADS, num_layers=NUM_LAYERS,
        contrastive_proj_dim=CONTRASTIVE_PROJ_DIM
    ).to(device)

    if torch.cuda.is_available() and torch.cuda.device_count() > 1:
        print(f"[*] Dùng {torch.cuda.device_count()} GPU với nn.DataParallel")
        model = nn.DataParallel(model)

    # Label smoothing: giảm overconfidence của model (một nguyên nhân gây hallucination).
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=LABEL_SMOOTHING)
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE, weight_decay=0.01)

    num_training_steps = len(train_loader) * EPOCHS
    num_warmup_steps = int(0.05 * num_training_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps
    )
    amp_enabled = USE_AMP and torch.cuda.is_available()
    scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)
    print(f"[*] Mixed precision (AMP): {'BAT' if amp_enabled else 'TAT'} (USE_AMP={USE_AMP}, cuda={torch.cuda.is_available()})")

    early_stop_mode = "min" if EARLY_STOP_METRIC == "loss" else "max"
    early_stopping = EarlyStoppingCallback(patience=5, save_path=BEST_MODEL_FILE, mode=early_stop_mode)

    # -------------------------
    # RESUME TỪ CHECKPOINT (chỉ để tiếp tục sau khi Kaggle kernel bị ngắt, không load baseline)
    # -------------------------
    start_epoch = load_checkpoint(
        model,
        optimizer,
        scheduler,
        early_stopping,
        filepath=RESUME_CHECKPOINT_FILE,
        resume_training=RESUME_TRAINING,
        reset_training_state=FINE_TUNE_RESET_TRAINING_STATE
    )

    log_interval = 200  # Bộ dữ liệu lớn, in thưa hơn để tránh trôi màn hình

    for epoch in range(start_epoch, EPOCHS):
        # Seed theo epoch giúp resume từ checkpoint vẫn tái lập đúng thứ tự batch/augmentation.
        epoch_seed = SEED + epoch
        seed_everything(epoch_seed)
        if train_loader.generator is not None:
            train_loader.generator.manual_seed(epoch_seed)
        print(f"\n--- Bắt đầu Epoch {epoch+1:02d}/{EPOCHS} ---")
        # USE_HALL_LOSS=False -> lambda=0 nên hall_loss vẫn được TÍNH (để log/theo dõi qua
        # object_token_ids dùng chung với Dynamic Intercept/Contrastive) nhưng KHÔNG cộng vào loss.
        current_lambda_hall = get_hall_lambda(epoch, LAMBDA_HALL, HALL_LAMBDA_WARMUP_EPOCHS) if USE_HALL_LOSS else 0.0
        hall_min_step_index = int(HALL_LOSS_MIN_STEP_RATIO * SEQ_LENGTH)
        print(f"   [Hall Loss] lambda={current_lambda_hall:.5f} (max={LAMBDA_HALL}) | min_step_index={hall_min_step_index}")

        # -------------------------
        # PHASE 1: TRAINING
        # -------------------------
        current_lambda_con = get_contrastive_lambda(epoch, LAMBDA_CONTRASTIVE, CONTRASTIVE_WARMUP_EPOCHS) if USE_CONTRASTIVE_LOSS else 0.0
        if USE_CONTRASTIVE_LOSS:
            print(f"   [Contrastive] lambda={current_lambda_con:.5f} (max={LAMBDA_CONTRASTIVE}) | tau={CONTRASTIVE_TAU}")

        model.train()
        train_loss = 0
        train_ce_loss = 0
        train_hall_loss = 0
        train_con_loss = 0
        train_correct = 0
        train_tokens = 0
        model_ = _unwrap_model(model)

        for batch_idx, (images, captions, image_paths, gt_objects_batch) in enumerate(train_loader):
            images, captions = images.to(device), captions.to(device)

            decoder_input = captions[:, :-1]
            targets = captions[:, 1:]

            optimizer.zero_grad()

            with torch.cuda.amp.autocast(enabled=amp_enabled):
                if USE_CONTRASTIVE_LOSS:
                    # Gọi thẳng encode_image/decode (bỏ qua DataParallel.forward) để dùng chung
                    # đúng 1 lần `memory` cho cả positive và negative caption - tránh forward lại
                    # CLIP-ViT (đóng băng nhưng vẫn tốn compute) 2 lần mỗi batch. Kaggle T4 là
                    # trường hợp phổ biến (1 GPU) nên bỏ qua DataParallel ở nhánh này chấp nhận được.
                    memory = model_.encode_image(images)
                    outputs, pos_hidden = model_.decode(memory, decoder_input, return_hidden=True)
                else:
                    outputs = model(images, decoder_input)
                # Intercept vào decoder NGAY LÚC TRAIN: hạ logit của token thuộc object vắng mặt
                # trước khi tính CE/hall_loss, để model học trực tiếp việc bị chặn thay vì chỉ
                # bị phạt gián tiếp qua hall_loss.
                outputs = apply_decoder_object_intercept(
                    outputs, image_paths, gt_objects_batch, object_token_ids,
                    targets=targets, epoch_index=epoch
                )
                ce_loss = criterion(outputs.reshape(-1, vocab_size), targets.reshape(-1))
                if USE_HALL_LOSS:
                    hall_loss = hallucination_object_loss(
                        outputs, image_paths, gt_objects_batch, object_token_ids, token_mask=(targets != PAD_IDX)
                    )
                else:
                    hall_loss = outputs.new_tensor(0.0)

                if USE_CONTRASTIVE_LOSS:
                    neg_captions_full = build_negative_captions(captions, gt_objects_batch, object_token_ids, id_to_obj_name)
                    neg_decoder_input = neg_captions_full[:, :-1]
                    _, neg_hidden = model_.decode(memory, neg_decoder_input, return_hidden=True)

                    image_embed = model_.project_image(memory)
                    pos_text_embed = model_.project_text(pos_hidden, decoder_input == PAD_IDX)
                    neg_text_embed = model_.project_text(neg_hidden, neg_decoder_input == PAD_IDX)
                    con_loss = contrastive_image_caption_loss(image_embed, pos_text_embed, neg_text_embed, tau=CONTRASTIVE_TAU)
                else:
                    con_loss = outputs.new_tensor(0.0)

                loss = ce_loss + (current_lambda_hall * hall_loss) + (current_lambda_con * con_loss)

            predictions = outputs.argmax(dim=-1)
            mask = (targets != PAD_IDX)
            correct = (predictions == targets) & mask

            train_correct += correct.sum().item()
            train_tokens += mask.sum().item()

            scaler.scale(loss).backward()
            # unscale truoc khi clip: clip phai tinh tren gradient that, khong phai gradient da scale boi AMP.
            scaler.unscale_(optimizer)
            # Gradient clipping: ổn định gradient khi train decoder từ đầu (không có baseline dẫn dắt).
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            train_loss += loss.item()
            train_ce_loss += ce_loss.item()
            train_hall_loss += hall_loss.item()
            train_con_loss += con_loss.item()

            if (batch_idx + 1) % log_interval == 0 or (batch_idx + 1) == len(train_loader):
                batch_acc = (correct.sum().item() / mask.sum().item()) * 100 if mask.sum().item() > 0 else 0.0
                print(f"   [Epoch {epoch+1:02d}] Batch {batch_idx+1:05d}/{len(train_loader)} | Loss: {loss.item():.4f} | CE: {ce_loss.item():.4f} | Hall: {hall_loss.item():.6f} | Con: {con_loss.item():.6f} | Acc: {batch_acc:.2f}%")

        epoch_train_loss = train_loss / len(train_loader)
        epoch_train_ce_loss = train_ce_loss / len(train_loader)
        epoch_train_hall_loss = train_hall_loss / len(train_loader)
        epoch_train_con_loss = train_con_loss / len(train_loader)
        epoch_train_acc = (train_correct / train_tokens) * 100

        # -------------------------
        # PHASE 2: VALIDATION
        # -------------------------
        model.eval()
        val_loss = 0
        val_ce_loss = 0
        val_hall_loss = 0
        val_con_loss = 0
        val_correct = 0
        val_tokens = 0

        if val_loader.generator is not None:
            val_loader.generator.manual_seed(SEED + 100_000 + epoch)

        with torch.no_grad():
            for images, captions, image_paths, gt_objects_batch in val_loader:
                images, captions = images.to(device), captions.to(device)

                decoder_input = captions[:, :-1]
                targets = captions[:, 1:]

                if USE_CONTRASTIVE_LOSS:
                    memory = model_.encode_image(images)
                    outputs, pos_hidden = model_.decode(memory, decoder_input, return_hidden=True)
                else:
                    outputs = model(images, decoder_input)
                # Giữ nhất quán với training: val_loss/val_acc phản ánh đúng phân phối "đã bị
                # chặn" mà model thực sự dùng lúc generate.
                outputs = apply_decoder_object_intercept(
                    outputs, image_paths, gt_objects_batch, object_token_ids,
                    targets=targets, epoch_index=epoch
                )
                ce_loss = criterion(outputs.reshape(-1, vocab_size), targets.reshape(-1))
                if USE_HALL_LOSS:
                    hall_loss = hallucination_object_loss(
                    outputs, image_paths, gt_objects_batch, object_token_ids, token_mask=(targets != PAD_IDX)
                )
                else:
                    hall_loss = outputs.new_tensor(0.0)

                if USE_CONTRASTIVE_LOSS:
                    neg_captions_full = build_negative_captions(captions, gt_objects_batch, object_token_ids, id_to_obj_name)
                    neg_decoder_input = neg_captions_full[:, :-1]
                    _, neg_hidden = model_.decode(memory, neg_decoder_input, return_hidden=True)
                    image_embed = model_.project_image(memory)
                    pos_text_embed = model_.project_text(pos_hidden, decoder_input == PAD_IDX)
                    neg_text_embed = model_.project_text(neg_hidden, neg_decoder_input == PAD_IDX)
                    con_loss = contrastive_image_caption_loss(image_embed, pos_text_embed, neg_text_embed, tau=CONTRASTIVE_TAU)
                else:
                    con_loss = outputs.new_tensor(0.0)

                loss = ce_loss + (current_lambda_hall * hall_loss) + (current_lambda_con * con_loss)
                val_loss += loss.item()
                val_ce_loss += ce_loss.item()
                val_hall_loss += hall_loss.item()
                val_con_loss += con_loss.item()

                predictions = outputs.argmax(dim=-1)
                mask = (targets != PAD_IDX)
                val_correct += ((predictions == targets) & mask).sum().item()
                val_tokens += mask.sum().item()

        epoch_val_loss = val_loss / len(val_loader)
        epoch_val_ce_loss = val_ce_loss / len(val_loader)
        epoch_val_hall_loss = val_hall_loss / len(val_loader)
        epoch_val_con_loss = val_con_loss / len(val_loader)
        epoch_val_acc = (val_correct / val_tokens) * 100

        # BLEU-4 thật trên subset val (greedy decode) - chỉ tính khi cần cho theo dõi/early-stop.
        epoch_val_bleu4 = None
        if EARLY_STOP_METRIC == "bleu4" and val_ref_map:
            epoch_val_bleu4 = evaluate_val_bleu4(model, vocab, val_ref_map, max_images=VAL_METRIC_MAX_SAMPLES)

        monitor_score = epoch_val_bleu4 if EARLY_STOP_METRIC == "bleu4" else epoch_val_loss

        # -------------------------
        # BÁO CÁO VÀ LƯU CHECKPOINT
        # -------------------------
        print(f"\n=> TỔNG KẾT EPOCH {epoch+1:02d}:")
        print(f"   Train Loss: {epoch_train_loss:.4f} | CE: {epoch_train_ce_loss:.4f} | Hall: {epoch_train_hall_loss:.6f} | Con: {epoch_train_con_loss:.6f} | Train Acc: {epoch_train_acc:.2f}%")
        bleu_str = f" | BLEU-4: {epoch_val_bleu4:.4f}" if epoch_val_bleu4 is not None else ""
        print(f"   Val Loss  : {epoch_val_loss:.4f} | CE: {epoch_val_ce_loss:.4f} | Hall: {epoch_val_hall_loss:.6f} | Con: {epoch_val_con_loss:.6f} | Val Acc  : {epoch_val_acc:.2f}%{bleu_str}")

        # 1. Lưu Best Model theo EARLY_STOP_METRIC (mặc định BLEU-4, không phải val_loss)
        early_stopping(monitor_score, model, optimizer, epoch)

        # 2. Luôn lưu Last Checkpoint sau mỗi epoch để Resume khi sập Kaggle kernel
        epoch_metrics = {
            "val_loss": epoch_val_loss,
            "val_ce_loss": epoch_val_ce_loss,
            "val_hall_loss": epoch_val_hall_loss,
            "val_con_loss": epoch_val_con_loss,
            "val_acc": epoch_val_acc,
            "val_bleu4": epoch_val_bleu4,
        }
        save_checkpoint(model, optimizer, scheduler, epoch, epoch_metrics, early_stopping, CHECKPOINT_FILE)

        if early_stopping.early_stop:
            print(f"⚠️ Dừng huấn luyện sớm tại Epoch {epoch+1} do {EARLY_STOP_METRIC} không cải thiện liên tiếp.")
            break

    return model, optimizer

In [ ]:
# ==========================================
# CHẨN ĐOÁN: FILENAME KARPATHY CÓ KHỚP VỚI COCO INSTANCES KHÔNG?
# ==========================================
def diagnose_hall_label_coverage(image_paths, image_objects, label="train"):
    """Cảnh báo sớm nguyên nhân PHỔ BIẾN NHẤT khiến hallucination_object_loss/
    apply_decoder_object_intercept không có tác dụng gì: HallDebug in ra `gt_images=0 |
    present_images=0` ở MỌI batch. Thường do TRAIN_INSTANCES_JSON/VAL_INSTANCES_JSON sai path,
    hoặc filename trong file instances không khớp filename ảnh thật (một số dataset Kaggle đổi
    tên file, bỏ tiền tố COCO_train2014_/COCO_val2014_)."""
    if not image_paths:
        return
    sample_basenames = [os.path.basename(p) for p in image_paths[:2000]]
    matched = sum(1 for name in sample_basenames if name in image_objects)
    coverage = matched / len(sample_basenames) * 100
    print(f"   [HallCheck-{label}] Khớp filename Karpathy <-> COCO instances: {matched}/{len(sample_basenames)} ({coverage:.1f}%) trên mẫu {len(sample_basenames)} ảnh đầu.")
    if coverage < 5:
        example_img = sample_basenames[0]
        example_ann = next(iter(image_objects.keys()), None)
        print(f"   [!] CẢNH BÁO: gần như không khớp filename -> hallucination_object_loss VÀ apply_decoder_object_intercept sẽ luôn = 0 (vô tác dụng), model đang train như baseline thường.")
        print(f"       Ví dụ tên ảnh Karpathy: '{example_img}'")
        print(f"       Ví dụ key trong COCO instances: '{example_ann}'")
        print(f"       -> Kiểm tra lại TRAIN_INSTANCES_JSON/VAL_INSTANCES_JSON đúng dataset khớp với ảnh đang dùng (tiền tố COCO_train2014_/COCO_val2014_, đuôi .jpg, v.v).")

# ==========================================
# THỰC THI (MAIN) - TRAIN FROM SCRATCH
# ==========================================
if __name__ == "__main__":
    print("0. Đang chuẩn bị Dataset và Vocabulary cho train-from-scratch COCO 2014...")
    print(f"   TRAIN_IMG_DIR  : {TRAIN_IMG_DIR}")
    print(f"   VAL_IMG_DIR    : {VAL_IMG_DIR}")
    print(f"   KARPATHY_JSON  : {KARPATHY_JSON_FILE}")
    print(f"   SEED={SEED} | cuDNN deterministic=True | benchmark=False")
    print(f"   FF_DIM={FF_DIM} | NUM_LAYERS={NUM_LAYERS} | LR={LEARNING_RATE} | EPOCHS={EPOCHS} | LABEL_SMOOTHING={LABEL_SMOOTHING}")
    print(f"   EARLY_STOP_METRIC={EARLY_STOP_METRIC} (VAL_METRIC_MAX_SAMPLES={VAL_METRIC_MAX_SAMPLES})")
    print(f"   TRAIN_DECODER_INTERCEPT={TRAIN_DECODER_INTERCEPT} | MODE={TRAIN_INTERCEPT_MODE} | MAX_PENALTY={TRAIN_INTERCEPT_MAX_PENALTY} | WARMUP={TRAIN_INTERCEPT_WARMUP_EPOCHS}")
    print(f"   USE_CONTRASTIVE_LOSS={USE_CONTRASTIVE_LOSS} | LAMBDA_CONTRASTIVE={LAMBDA_CONTRASTIVE} | TAU={CONTRASTIVE_TAU} | WARMUP={CONTRASTIVE_WARMUP_EPOCHS}")

    # 0.1 Load danh sách ảnh và caption tập Train theo Karpathy split
    train_imgs, train_caps = load_karpathy_data(KARPATHY_JSON_FILE, split="train")

    # 0.2 Load danh sách ảnh và caption tập Val theo Karpathy split
    val_imgs, val_caps = load_karpathy_data(KARPATHY_JSON_FILE, split="val")

    # 0.3 Xây dựng (hoặc Tải) Vocabulary từ tập Train và lưu ra file (đảm bảo file có thể load lại)
    vocab = build_or_load_vocab(train_caps, min_freq=MIN_WORD_FREQ, vocab_file=VOCAB_FILE)
    vocab_size = len(vocab)

    # USE_OBJECT_GROUNDING = USE_HALL_LOSS or TRAIN_DECODER_INTERCEPT or USE_CONTRASTIVE_LOSS:
    # dựng hạ tầng object/COCO-instances khi BẤT KỲ thành phần nào cần, không chỉ riêng HalluLoss
    # (Dynamic Decoder Intercept và Contrastive loss dùng chung dữ liệu này).
    object_token_ids = build_object_token_ids(vocab) if USE_OBJECT_GROUNDING else {}
    train_image_objects = load_coco_instance_objects(TRAIN_INSTANCES_JSON) if (USE_OBJECT_GROUNDING and USE_COCO_INSTANCE_HALL_LABELS) else {}
    val_image_objects = load_coco_instance_objects(VAL_INSTANCES_JSON) if (USE_OBJECT_GROUNDING and USE_COCO_INSTANCE_HALL_LABELS) else {}
    if USE_OBJECT_GROUNDING:
        print(f"   Object grounding: USE_HALL_LOSS={USE_HALL_LOSS} (lambda={LAMBDA_HALL}) | "
              f"TRAIN_DECODER_INTERCEPT={TRAIN_DECODER_INTERCEPT} | USE_CONTRASTIVE_LOSS={USE_CONTRASTIVE_LOSS} | "
              f"COCO instances (ground truth only, không dùng YOLO)={USE_COCO_INSTANCE_HALL_LABELS}")
        if USE_COCO_INSTANCE_HALL_LABELS:
            diagnose_hall_label_coverage(train_imgs, train_image_objects, label="train")
            diagnose_hall_label_coverage(val_imgs, val_image_objects, label="val")

    # Contrastive loss (research/contrastive-loss-for-caption-hallucination.md) tái dùng đúng
    # object_token_ids đã build ở trên (qua USE_OBJECT_GROUNDING) - cần object_token_ids không rỗng.
    id_to_obj_name = build_object_id_to_name(object_token_ids) if USE_CONTRASTIVE_LOSS else {}
    if USE_CONTRASTIVE_LOSS:
        print(f"   Contrastive loss: lambda={LAMBDA_CONTRASTIVE}, tau={CONTRASTIVE_TAU}, "
              f"warmup_epochs={CONTRASTIVE_WARMUP_EPOCHS}, proj_dim={CONTRASTIVE_PROJ_DIM}")
        if not object_token_ids:
            print("   [!] CẢNH BÁO: object_token_ids rỗng -> build_negative_captions sẽ không có "
                  "object nào để tạo negative, contrastive loss sẽ luôn ~0 (vô tác dụng).")

    # 0.4 Kiểm tra pycocoevalcap trước khi train (fail-fast) - nếu thiếu thì fallback về "loss"
    # thay vì crash giữa chừng epoch đầu tiên khi evaluate_val_bleu4 được gọi.
    if EARLY_STOP_METRIC == "bleu4":
        try:
            from pycocoevalcap.bleu.bleu import Bleu  # noqa: F401
        except ImportError:
            print("[!] Không tìm thấy pycocoevalcap -> fallback EARLY_STOP_METRIC='loss'. "
                  "Cài `pip install pycocoevalcap` để dùng BLEU-4 chuẩn COCO caption cho early-stopping.")
            EARLY_STOP_METRIC = "loss"

    # 0.5 Reference map (nhiều caption / ảnh) để tính BLEU-4 thật mỗi epoch, nếu được bật.
    val_ref_map = None
    if EARLY_STOP_METRIC == "bleu4":
        val_ref_map = build_val_reference_map(val_imgs, val_caps, max_images=VAL_METRIC_MAX_SAMPLES)
        print(f"   -> Sẽ tính BLEU-4 mỗi epoch trên {len(val_ref_map)} ảnh val (greedy decode).")

    # 1. Khởi tạo Dataset & DataLoader
    train_dataset = COCODataset(train_imgs, train_caps, vocab, transforms=train_transforms, image_objects=train_image_objects)
    val_dataset = COCODataset(val_imgs, val_caps, vocab, transforms=val_transforms, image_objects=val_image_objects)

    print("1. Đang khởi tạo DataLoaders...")
    train_generator = torch.Generator()
    train_generator.manual_seed(SEED)
    val_generator = torch.Generator()
    val_generator.manual_seed(SEED + 1)
    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=4, pin_memory=True if torch.cuda.is_available() else False,
        worker_init_fn=seed_worker, generator=train_generator
    )
    val_loader = DataLoader(
        val_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=4, pin_memory=True if torch.cuda.is_available() else False,
        worker_init_fn=seed_worker, generator=val_generator
    )

    print("2. Bắt đầu train CLIP-ViT-B/16 + Transformer TỪ ĐẦU với hallucination-aware loss trên COCO 2014...")
    model, optimizer = train_model(
        train_loader,
        val_loader,
        vocab=vocab,
        object_token_ids=object_token_ids,
        val_ref_map=val_ref_map,
        id_to_obj_name=id_to_obj_name,
    )

    print("3. Quá trình huấn luyện hoàn tất toàn bộ số epochs yêu cầu.")